# Stage 5.05 — validate, analyze, and export

Run validation and analysis for both 5A and 5B. When the 5A0 gate is closed, the analysis produces the selected operating point and the final observations without any episode results.

In [ ]:
import json, os, subprocess
from pathlib import Path
import sys

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"
ANALYSIS.mkdir(exist_ok=True)
PY = Path(sys.executable).resolve()

# 5A validation
subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.validate_stage5",
    "--phase", "a",
    "--manifest", str(OUT / "stage5a_manifest.csv"),
    "--output-dir", str(OUT),
    "--audit", str(OUT / "stage5_openvla_coverage_capability_audit.json"),
], cwd=R, check=True)

# 5A analysis
subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.analyze_stage5",
    "--phase", "a",
    "--results", str(OUT / "stage5a_episode_results.csv"),
    "--output-dir", str(ANALYSIS),
], cwd=R, check=True)


In [ ]:
# 5B validation and analysis (conditional; no-ops when the gate is closed)
import os, subprocess, sys
from pathlib import Path
R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.validate_stage5",
    "--phase", "b",
    "--manifest", str(OUT / "stage5b_manifest.csv"),
    "--output-dir", str(OUT),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
], cwd=R, check=True)

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.analyze_stage5",
    "--phase", "b",
    "--results", str(OUT / "stage5b_episode_results.csv"),
    "--output-dir", str(ANALYSIS),
], cwd=R, check=True)


In [ ]:
import hashlib
from pathlib import Path
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"

expected = ["stage5a_selected_operating_point.json", "STAGE_5A_OBSERVATIONS.md"]
missing = [x for x in expected if not (ANALYSIS / x).exists()]
if missing:
    raise SystemExit(f"missing analysis artifacts: {missing}")

print("Stage 5 analysis artifacts:")
for p in sorted(ANALYSIS.iterdir()):
    print(" ", p.name)
